# ProbabilityFunction

The ProbabilityFunction class implements the reconstruction of random variables' PDFs.\
For instance, it can be used to estimate the photo-$z$s of individual galaxy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fomoz.data import Dataset
from fomoz.probability import ProbabilityFunction

## Input samples

For demonstration purposes, we use the same catalog as both the reference and test samples. This illustrates the workflow, not performance on an independent sample. In realistic scenarios, reference and test samples must be independent.

Magnitude errors must be present in the input catalogs, because they define the matching between reference and test objects. Here, we assume 0.1 mag errors.

In [ ]:
# Load the sample and assign dummy magnitude errors
sample = pd.read_csv('../data/input/reference.csv', comment='#')
bands = ['g', 'r', 'i']

for band in bands:
    if f'{band}_error' not in sample:
        sample[f'{band}_error'] = 0.1
        print(f'{band}: assuming 0.1 mag uncertainty for this example.')

# Dataset
reference = Dataset(sample, bands, z_gal_string='redshift')

## Redshift PDFs

`delta` sets the redshift sampling step. `knot_spacing` sets the spline knot spacing.

In [ ]:
pdf = ProbabilityFunction(
    reference_dataset=reference,
    test_dataset=reference,  # in a realistic scenario, this should be an actual test sample (see above)
    random_variable='redshift',
    delta=0.02,
    min_variable=0.001,
    max_variable=1.5,
)

pdf.select_features_grid()  # select the most informative colors at each sampled PDF point

pdf.process(cpu_number=1, knot_spacing=0.05)

# Get the original DataFrame, which also includes the following keys:
# fomoz_prob: Weighted membership probability.
# fomoz_prob_error: Approximate sampling standard error of the probability.
# fomoz_n_good: Sum of weighted memberships over matching references, i.e. the numerator in fomoz_prob.
# fomoz_n_all: Unweighted number of matching reference objects.
# fomoz_n_colors_used: Number of usable selected colors for each test object.
results = pdf.get_results_dataframe()

## Photo-z of one galaxy

Show a successful fit, its median, and its 68% interval, compared with the catalog redshift.

In [ ]:
print("\nReminder: this example is based only on 3 photometric bands (g, r, i) and a small reference sample\n")

i = 15

z16, z50, z84 = results.iloc[i][['fomoz_p16', 'fomoz_median', 'fomoz_p84']]
print(f'Photo-z: {z50:.3f}; 68% interval: [{z16:.3f}, {z84:.3f}]')

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(pdf.variable_samples, pdf.probability_functions[i], color='royalblue', label='FOMO-Z PDF')
ax.axvspan(z16, z84, color='royalblue', alpha=0.15, label='68% interval')
ax.axvline(z50, color='royalblue', linestyle='--', label='Photo-z median')
ax.axvline(sample['redshift'].iloc[i], color='black', linestyle=':', label='Catalog redshift')
ax.set_xlabel('Redshift', fontsize=13)
ax.set_ylabel('Probability density', fontsize=13)
ax.legend(fontsize=11)
plt.show()